# Tiny-GPT: GPT-6 Modern Architecture & FineWeb-Edu Training

**What's new in GPT-6:**
- **Tokenizer:** Byte-Level BPE (`8,192` vocab, zero `[UNK]`, lossless spacing)
- **Architecture:** RoPE + RMSNorm + SwiGLU + Weight Tying + Flash/SDPA + FP16 AMP
- **Dataset:** `HuggingFaceFW/fineweb-edu` (`sample-10BT`, `200M` tokens streamed to `uint16` `.bin`)
- **Safety:** Auto-saves checkpoints to Google Drive & supports `auto_resume: true`!

In [ ]:
REPO_URL = "https://github.com/nandish029/tiny-gpt.git"
BRANCH = "feature/gpt4-architecture-experiments"

RUN_ABLATION = True      # Compare Classic vs Modern architecture (2,000 steps each)
RUN_GPT6_FULL = False    # Set True to train 8L D=384 GPT-6 for 25,000 steps
TARGET_TRAIN_TOKENS = 200_000_000
DRIVE_DIR = "/content/drive/MyDrive/tiny-gpt-gpt6"

In [ ]:
# === STAGE 1: GPU CHECK & GOOGLE DRIVE MOUNT ===
import os, torch
assert torch.cuda.is_available(), "ERROR: Select Runtime -> Change runtime type -> T4 GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# === STAGE 2: CLONE REPO & INSTALL DEPS ===
%cd /content
!rm -rf tiny-gpt
!git clone --branch {BRANCH} {REPO_URL} tiny-gpt
%cd /content/tiny-gpt
!pip -q install datasets tokenizers pyyaml numpy

In [ ]:
# === STAGE 3: STREAM FINEWEB-EDU (CACHE TO DRIVE SO YOU ONLY STREAM ONCE) ===
import os, shutil
drive_data = f"{DRIVE_DIR}/data_fineweb_edu"
if os.path.isfile(f"{drive_data}/train.bin") and os.path.isfile(f"{drive_data}/fineweb_bpe_8192.json"):
    os.makedirs("data/fineweb_edu", exist_ok=True)
    for fn in ["train.bin", "val.bin", "fineweb_bpe_8192.json", "dataset_manifest.json"]:
        shutil.copy2(f"{drive_data}/{fn}", f"data/fineweb_edu/{fn}")
    print("Restored FineWeb-Edu cache from Google Drive!")
else:
    !python -u src/data/prepare_fineweb.py --target_train_tokens {TARGET_TRAIN_TOKENS}
    shutil.copytree("data/fineweb_edu", drive_data, dirs_exist_ok=True)
    print(f"Cached FineWeb-Edu to {drive_data}")

In [ ]:
# === STAGE 4: ARCHITECTURE ABLATION (CLASSIC vs MODERN ON FINEWEB-EDU) ===
import json
from src.experiments.runner import ExperimentRunner
from src.generation.generate import generate

if RUN_ABLATION:
    ablation_rows = []
    for cfg in ["experiments/gpt6/configs/ablation_classic.yaml", "experiments/gpt6/configs/ablation_modern.yaml"]:
        print(f"\n=== {cfg} ===")
        r = ExperimentRunner(cfg, device="cuda")
        params = r.count_parameters()["trainable"]
        res = r.train(overwrite=True)
        ev = r.evaluate(batch_count=50, split="val", seed=10042)
        ablation_rows.append({
            "name": r.name, "params": params, "time_s": res["elapsed_seconds"],
            "min_val": res["min_val_loss"], "eval_loss": ev["loss"], "eval_ppl": ev["perplexity"]
        })
    print("\n================ COPY ABLATION TABLE TO CHAT ================")
    for row in ablation_rows:
        print(row)

In [ ]:
# === STAGE 5: TRAIN GPT-6 FULL (AUTO-RESUMES FROM GOOGLE DRIVE) ===
if RUN_GPT6_FULL:
    import shutil, zipfile, json
    drive_ckpt = f"{DRIVE_DIR}/gpt6_full_baseline.pt"
    if os.path.isfile(drive_ckpt):
        os.makedirs("checkpoints/gpt6_full", exist_ok=True)
        shutil.copy2(drive_ckpt, "checkpoints/gpt6_full/gpt6_full_baseline.pt")
        print("Found existing checkpoint on Drive -> resuming!")

    r = ExperimentRunner("experiments/gpt6/configs/gpt6_full.yaml", device="cuda")
    res = r.train()
    shutil.copy2(r.checkpoint_path, drive_ckpt)
    ev = r.evaluate(batch_count=100, split="val", seed=10042)

    samples = {}
    for p in ["The history of science", "In Python, a function", "Once upon a time"]:
        gen = torch.Generator(device="cuda").manual_seed(42)
        samples[p] = generate(r.model, r.tokenizer, p, max_new_tokens=80, temperature=0.8, top_k=40, top_p=0.95, repetition_penalty=1.15, generator=gen)
        print(f"[{p}] -> {samples[p]}\n")

    with open("experiments/gpt6/results/gpt6_full_eval.json", "w") as f:
        json.dump({"train": res, "eval": ev, "samples": samples}, f, indent=2)

    out_zip = f"{DRIVE_DIR}/gpt6_colab_results.zip"
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(r.checkpoint_path)
        zf.write("experiments/gpt6/results/gpt6_full_eval.json")
        zf.write("data/fineweb_edu/dataset_manifest.json")
        zf.write("data/fineweb_edu/fineweb_bpe_8192.json")
    print(f"DONE! Saved {out_zip}")